# Phase 1 ANC calibration targets

Calculate initial delivery-location targets used in Phase 1 ANC calibration for any county with complete `base_LB` rows in the parameter workbook. The notebook does not edit the workbook. Review the calculated rows, then enter them manually in the `calibration_targets` sheet of `SDR Parameters.xlsx`.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

WORKBOOK = Path("SDR Parameters.xlsx")
if not WORKBOOK.exists():
    WORKBOOK = Path("data/SDR Parameters.xlsx")
if not WORKBOOK.exists():
    raise FileNotFoundError("Run this notebook from the repository root or data folder.")

county_supply = pd.read_excel(WORKBOOK, sheet_name="county_supply")

## Assumptions

For county $c$ and facility level $j$, first convert the observed monthly live births (`base_LB`) into final delivery-location shares:

$$F_{c,j} = \frac{base\_LB_{c,j}}{\sum_{k \in \{home,L2/3,L4,L5\}} base\_LB_{c,k}}$$

The county's estimated internal emergency-transfer share from L2/3 is the reference transfer share scaled by the county-to-reference ratio of final L2/3 delivery shares:

$$T_c = T_{ref} \times \frac{F_{c,L2/3}}{F_{ref,L2/3}}$$

where the default reference is Kakamega and $T_{ref}=0.122$. Let $\alpha$ be the proportion of these transfers going to L4; the remaining $1-\alpha$ go to L5. The default is $\alpha=0.5$. The initial delivery-location targets are reconstructed as:

$$I_{c,home} = F_{c,home}$$

$$I_{c,L2/3} = F_{c,L2/3} + T_c$$

$$I_{c,L4} = F_{c,L4} - \alpha T_c$$

$$I_{c,L5} = F_{c,L5} - (1-\alpha)T_c$$

Therefore, $\sum_j I_{c,j}=1$. The function checks this identity and rejects negative targets. The caller separately identifies which levels are calibration objectives and which are diagnostics; for Mombasa, L5 is diagnostic because its observed deliveries include external referrals.

In [9]:
FACILITY_LEVELS = ["home", "L2/3", "L4", "L5"]

In [10]:
def final_delivery_shares(county, supply):
    rows = supply.loc[
        supply["parameter_name"].eq("base_LB")
        & supply["county"].str.lower().eq(county.lower()),
        ["facility_level", "value"],
    ].copy()
    rows["facility_level"] = pd.Categorical(
        rows["facility_level"], categories=FACILITY_LEVELS, ordered=True
    )
    rows = rows.sort_values("facility_level")
    if rows["facility_level"].astype(str).tolist() != FACILITY_LEVELS:
        raise ValueError(f"Expected one base_LB row for each facility level in {county}.")
    births = rows["value"].astype(float).to_numpy()
    if not np.isfinite(births).all() or births.sum() <= 0:
        raise ValueError(f"The base_LB values for {county} must be finite and sum to more than zero.")
    return pd.Series(births / births.sum(), index=FACILITY_LEVELS, name=county.lower())

In [11]:
def calculate_phase1_anc_targets(
    county,
    supply,
    *,
    reference_county="kakamega",
    reference_transfer_share=0.122,
    transfer_to_l4_share=0.5,
    diagnostic_levels=("L5",),
):
    """Return calculation checks and workbook-ready Phase 1 targets."""
    if not 0 <= reference_transfer_share <= 1:
        raise ValueError("reference_transfer_share must be between 0 and 1.")
    if not 0 <= transfer_to_l4_share <= 1:
        raise ValueError("transfer_to_l4_share must be between 0 and 1.")
    unknown_diagnostics = set(diagnostic_levels) - set(FACILITY_LEVELS)
    if unknown_diagnostics:
        raise ValueError(f"Unknown diagnostic facility levels: {sorted(unknown_diagnostics)}")

    reference_final = final_delivery_shares(reference_county, supply)
    county_final = final_delivery_shares(county, supply)
    transfer_share = (
        reference_transfer_share
        * county_final["L2/3"]
        / reference_final["L2/3"]
    )
    transfer_to_l4 = transfer_share * transfer_to_l4_share
    transfer_to_l5 = transfer_share * (1 - transfer_to_l4_share)

    initial_by_level = pd.Series(
        {
            "home": county_final["home"],
            "L2/3": county_final["L2/3"] + transfer_share,
            "L4": county_final["L4"] - transfer_to_l4,
            "L5": county_final["L5"] - transfer_to_l5,
        },
        name="value",
    )
    if not np.isclose(initial_by_level.sum(), 1.0):
        raise ValueError("Calculated initial delivery-location shares do not sum to 1.")
    if (initial_by_level < 0).any():
        raise ValueError("At least one calculated initial delivery-location share is negative.")

    calculation_check = pd.DataFrame(
        {
            f"{county.title()} final share": county_final,
            f"{county.title()} initial target": initial_by_level,
        }
    )
    calculation_check.loc["total"] = calculation_check.sum()

    target_names = {
        "home": "home_all_target",
        "L2/3": "l23_all_target",
        "L4": "l4_all_target",
        "L5": "l5_all_target",
    }
    targets_for_workbook = initial_by_level.rename_axis("facility_level").reset_index()
    targets_for_workbook.insert(0, "target_name", targets_for_workbook["facility_level"].map(target_names))
    targets_for_workbook.insert(1, "county", county.lower())
    targets_for_workbook["role"] = targets_for_workbook["facility_level"].map(
        lambda level: "diagnostic" if level in diagnostic_levels else "objective"
    )
    targets_for_workbook["note"] = (
        f"Initial location target reconstructed from final base_LB shares; "
        f"{reference_transfer_share:.3f} {reference_county.title()} transfer relationship "
        f"scaled to {county.title()} and allocated "
        f"{transfer_to_l4_share:.0%} to L4 / {1 - transfer_to_l4_share:.0%} to L5."
    )
    return calculation_check, targets_for_workbook

In [12]:
calculation_check, mombasa_targets = calculate_phase1_anc_targets(
    county="mombasa",
    supply=county_supply,
    reference_county="kakamega",
    reference_transfer_share=0.122,
    transfer_to_l4_share=0.5,
    diagnostic_levels=("L5",),
)
mombasa_targets

,target_name,county,facility_level,value,role,note
0,home_all_target,mombasa,home,0.039001,objective,Initial location target reconstructed from fin...
1,l23_all_target,mombasa,L2/3,0.315242,objective,Initial location target reconstructed from fin...
2,l4_all_target,mombasa,L4,0.480812,objective,Initial location target reconstructed from fin...
3,l5_all_target,mombasa,L5,0.164946,diagnostic,Initial location target reconstructed from fin...


In [13]:
calculation_check, kisii_targets = calculate_phase1_anc_targets(
    county="kisii",
    supply=county_supply,
    reference_county="kakamega",
    reference_transfer_share=0.122,
    transfer_to_l4_share=0.5,
    diagnostic_levels=("L5",),
)
kisii_targets

,target_name,county,facility_level,value,role,note
0,home_all_target,kisii,home,0.058800,objective,Initial location target reconstructed from fin...
1,l23_all_target,kisii,L2/3,0.255115,objective,Initial location target reconstructed from fin...
2,l4_all_target,kisii,L4,0.484270,objective,Initial location target reconstructed from fin...
3,l5_all_target,kisii,L5,0.201816,diagnostic,Initial location target reconstructed from fin...


In [14]:
calculation_check, makueni_targets = calculate_phase1_anc_targets(
    county="makueni",
    supply=county_supply,
    reference_county="kakamega",
    reference_transfer_share=0.122,
    transfer_to_l4_share=0.5,
    diagnostic_levels=("L5",),
)
makueni_targets

,target_name,county,facility_level,value,role,note
0,home_all_target,makueni,home,0.090901,objective,Initial location target reconstructed from fin...
1,l23_all_target,makueni,L2/3,0.270035,objective,Initial location target reconstructed from fin...
2,l4_all_target,makueni,L4,0.537837,objective,Initial location target reconstructed from fin...
3,l5_all_target,makueni,L5,0.101227,diagnostic,Initial location target reconstructed from fin...
